In [2]:
# ============================================================
# STEP 8 — Explainability (SHAP)
# Shows which words triggered the phishing verdict.
# Red tokens  → push toward PHISHING
# Blue tokens → push toward SAFE
# ============================================================

import os
import torch
!pip install transformers
from transformers import pipeline
from google.colab import drive

# ============================================================
# 8.0  Mount Drive
# ============================================================
drive.mount("/content/drive", force_remount=True)

SAVE_DIR = "/content/drive/MyDrive/PhishingClassifier"
device   = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ============================================================
# 8.1  Install SHAP
# ============================================================
os.system("pip install -q shap")
import shap
print("SHAP ready.\n")

# ============================================================
# 8.2  Load Model via Pipeline
# ============================================================
print("Loading model...")
clf = pipeline(
    "text-classification",
    model=f"{SAVE_DIR}/model",
    tokenizer=f"{SAVE_DIR}/model",
    device=0 if torch.cuda.is_available() else -1,
    truncation=True,
    max_length=256,
    return_all_scores=True
)
print(f"  Ready on {device}\n")

# ============================================================
# 8.3  Run SHAP Explanations
# ============================================================
explainer = shap.Explainer(clf)

emails = [
    "URGENT: Your PayPal account has been limited! "
    "Click http://paypa1-secure.ru/verify to restore access now.",

    "Hi Sarah, confirming our 3pm meeting Thursday. "
    "I will bring the Q3 report. See you then.",

    "Dear customer, unusual activity detected on your bank account. "
    "Verify your identity at http://secure-bank-alert.com immediately."
]

print("=" * 55)
print("SHAP EXPLANATIONS")
print("  Red   → pushes toward PHISHING")
print("  Blue  → pushes toward SAFE")
print("=" * 55)

for i, email in enumerate(emails):
    print(f"\nEmail {i+1}: {email[:75]}...")
    vals = explainer([email])
    shap.plots.text(vals[0])

    # Top 5 most influential tokens
    values = vals[0].values
    tokens = vals[0].data
    skip   = {"[CLS]", "[SEP]", "[PAD]"}

    if len(values.shape) > 1:
        scored = [(t, v[1]) for t, v in zip(tokens, values) if t not in skip]
    else:
        scored = [(t, v) for t, v in zip(tokens, values) if t not in skip]

    scored.sort(key=lambda x: abs(x[1]), reverse=True)
    print("\n  Top 5 influential tokens:")
    for tok, val in scored[:5]:
        direction = "PHISHING" if val > 0 else "SAFE"
        print(f"    '{tok}' : {val:+.4f}  →  {direction}")

print("\nStep 8 complete.")

Mounted at /content/drive
SHAP ready.

Loading model...


Loading weights:   0%|          | 0/201 [00:01<?, ?it/s]

  Ready on cuda

SHAP EXPLANATIONS
  Red   → pushes toward PHISHING
  Blue  → pushes toward SAFE

Email 1: URGENT: Your PayPal account has been limited! Click http://paypa1-secure.ru...


You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset
PartitionExplainer explainer: 2it [00:10, 10.17s/it]               



  Top 5 influential tokens:
    '! ' : +0.0142  →  PHISHING
    'limited' : -0.0066  →  SAFE
    'been ' : -0.0061  →  SAFE
    'has ' : -0.0032  →  SAFE
    'Pay' : +0.0031  →  PHISHING

Email 2: Hi Sarah, confirming our 3pm meeting Thursday. I will bring the Q3 report. ...


...

PartitionExplainer explainer: 2it [00:10, 10.88s/it]               



  Top 5 influential tokens:
    'report' : -0.2216  →  SAFE
    'Thursday' : -0.1191  →  SAFE
    'pm ' : -0.0989  →  SAFE
    'our ' : -0.0890  →  SAFE
    '3 ' : -0.0785  →  SAFE

Email 3: Dear customer, unusual activity detected on your bank account. Verify your ...


  0%|          | 0/498 [00:00<?, ?it/s]

PartitionExplainer explainer: 2it [00:11, 11.13s/it]               



  Top 5 influential tokens:
    'bank ' : +0.0035  →  PHISHING
    'your ' : +0.0031  →  PHISHING
    'your ' : +0.0020  →  PHISHING
    'bank' : +0.0019  →  PHISHING
    'at ' : -0.0018  →  SAFE

Step 8 complete.
